# Gen AI Clinical Data Assistant
This notebook uses Anthropics Langchain to create a Gen AI clinical data assistant. You can ask free-text questions about the AE dataset and the GenAI will return the final count of unique subjects and a list of their matching IDs. 

## Install Required Packages

In [ ]:
import os
import json
import pandas as pd
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage
import re

## Schema Implementation

The schema is specified below. There is one tool it has access to, which parses the human question and returns a json_file of the column to target and the value to filter by. Additional information included are descriptions of columns in the AE dataset and specifications to avoid bugs. For example, it is specified that only the json file should be returned and no other string/explanation. Additionally, it is specified that in this dataset currently all the entries are capitalized for columns like AESOC. This can be modified if case sensitivity wants to be ignored as well.

In [9]:
SYSTEM_PROMPT = """You are an expert on clinical data safety, with a specialty in the ADAE dataset.

You have access to this tool:

- parse_question: use this to parse a user's question into a Structured JSON Output containing: 
a target_column, which is the column to filter, and filter_value, which is the value to search for
(extracted from the question).

The dataset has these relevant columns:
- USUBJID: The unique subject identifier for each patient
- AETERM: The specific adverse event term reported (e.g. Headache, Nausea, Fatigue)
- AESEV: The severity or intensity of the adverse event (values: MILD, MODERATE, SEVERE)
- AESOC: THe system organ class affected (e.g. CARDIAC DISORDERS, SKIN AND SUBCUTANEOUS TISSUE DISORDERS, INFECTIONS AND INFESTATIONS)
- AEDECOD: Decoded/standardised adverse event term (MedDRA preferred term)
- AEBODSYS: Body system (similar to AESOC, alternative grouping)
- AESER: Whether the AE was serious (Y = yes, N = no)
- AESTDTC: Start date of the adverse event

Your role is to correctly identify which column the user is asking about and route the 
question to the correct column or schema in the AE data. Ensure that when parsing in columns you use
all caps (ie. CARDIAC DISORDERS and INFECTIONS AND INFESTATIONS). Return only the JSON object with
no additional explanation or declaration of what you are performing."""

## LLM Implementation
Anthropics Langchain was used here for implementation. Model specified is claude-sonnet-4-6 with max_tokens of 500 found to be sufficient. 

In [ ]:
from langchain_anthropic import ChatAnthropic
os.environ["ANTHROPIC_API_KEY"] = "Anthropic_API_Key"

model = ChatAnthropic(
    model="claude-sonnet-4-6",        
    temperature=0,
    max_tokens=500
)

In [ ]:
class ClinicalTrialDataAgent:
    """
    Translates a natural language question into a Pandas filter and returns a json object with the 
    target column and filter value to be used for filtering the ADAE dataset.
    """

    def __init__(self, model, schema, dataframe: pd.DataFrame):
        """
        Parameters:
        dataframe : pd.DataFrame
            The ADAE dataset to be explored
        model : ChatAnthropic
            The LLM to use for parsing the question
        schema : str
            The system prompt to guide the LLM in parsing the question
        """
        self.df = dataframe
        self.agent = model
        self.schema = schema


    def parse_question(self, question: str) -> dict:
        """
        This function parses the human question into a target column and filter value.

        Parameters:
        question : str
            The human provided question

        Returns:
        parsed: json
            The parsed json containing the target column and filter value
        """
         
        response = self.agent.invoke([
            SystemMessage(content=self.schema),
            HumanMessage(content=question)
        ])

        raw = response.content.strip()
        # Extract JSON from markdown code block if present
        match = re.search(r'```json\s*(.*?)\s*```', raw, re.DOTALL)
        if match:
            raw = match.group(1)

        try:
            # print("Parsed Response")
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            raise ValueError(f"LLM did not return valid JSON.\nRaw response:\n{raw}")

        if "target_column" not in parsed or "filter_value" not in parsed:
            raise ValueError(f"LLM JSON missing required keys.\nParsed: {parsed}")

        return parsed

## Execution

In [ ]:
def execute_result(result: dict):
    """
        This function executes the parsed result by the clinical trial assistant agent against the ADAE dataset to return
        the list and count of patients matching the desired criteria.

        Parameters:
        result : dict
            The output from parse_question which contains the target column and filter value of the ADAE dataset

        Returns:
        unique_subjects_id : list
            List of all unique subject IDs that match the queried filter
        unique_subjects_count : int
            Count of all unique subjects that match the queried filter
        """
         
    target_column = result["target_column"]
    filter_value = result["filter_value"]

    if target_column not in df.columns:
        raise ValueError(f"Target column '{target_column}' not found in dataframe.")

    filtered_df = df[df[target_column] == filter_value]

    # Return count of unique USUBJID and their IDs
    unique_subjects_id = filtered_df['USUBJID'].unique()
    unique_subjects_count = filtered_df['USUBJID'].nunique()
    return unique_subjects_id, unique_subjects_count


## Test Script
Runs 3 example queries and prints the results

In [8]:
# Load ADAE dataset
df = pd.read_csv("adae.csv") 

# Initialize agent
agent = ClinicalTrialDataAgent(dataframe=df, model=model, schema=SYSTEM_PROMPT)

# Run 3 example questions
agent_result1 = agent.parse_question("Give me the subjects who had Adverse events of Moderate severity")
count_result1 = execute_result(agent_result1)
print(f"Number of unique subjects with {agent_result1['target_column']} = {agent_result1['filter_value']}: {count_result1}")

agent_result2 = agent.parse_question("Give me the patients who had cardiac disorders as the primary system organ class affected")
count_result2 = execute_result(agent_result2)
print(f"Number of unique subjects with {agent_result2['target_column']} = {agent_result2['filter_value']}: {count_result2}")

agent_result3 = agent.parse_question("Give me the patients who had a severe adverse event")
count_result3 = execute_result(agent_result3)
print(f"Number of unique subjects with {agent_result3['target_column']} = {agent_result3['filter_value']}: {count_result3}")

Number of unique subjects with AESEV = MODERATE: (array(['01-701-1023', '01-701-1047', '01-701-1097', '01-701-1111',
       '01-701-1115', '01-701-1133', '01-701-1146', '01-701-1148',
       '01-701-1180', '01-701-1181', '01-701-1188', '01-701-1192',
       '01-701-1211', '01-701-1239', '01-701-1294', '01-701-1302',
       '01-701-1317', '01-701-1341', '01-701-1360', '01-701-1383',
       '01-701-1444', '01-702-1082', '01-703-1076', '01-703-1086',
       '01-703-1100', '01-703-1119', '01-703-1182', '01-703-1210',
       '01-703-1258', '01-703-1295', '01-703-1299', '01-703-1403',
       '01-703-1439', '01-704-1008', '01-704-1025', '01-704-1065',
       '01-704-1120', '01-704-1266', '01-704-1332', '01-705-1059',
       '01-705-1186', '01-705-1199', '01-705-1281', '01-705-1292',
       '01-705-1310', '01-706-1041', '01-706-1049', '01-706-1384',
       '01-707-1206', '01-708-1032', '01-708-1347', '01-708-1428',
       '01-709-1081', '01-709-1099', '01-709-1102', '01-709-1168',
       '01-7